# A pipeline to create a new WestWise dataset from pre-existing data

In [6]:
import re
from datetime import datetime

import ee

# -----------------------------------------------------------------------------
# USER PARAMETERS
# -----------------------------------------------------------------------------

GEE_PROJECT = "ee-tymc5571-cbi-module"
PROJECT_DIR = "projects/ee-tymc5571-cbi-module/assets"

PERIM_STRING = "welty_test_nocbi"
FIRES_ASSET = f"{PROJECT_DIR}/welty_wildfire_west_1984_2020"
YEAR_ATTR = "Fire_Year"

PRECOMPUTED_COLLECTIONS = [
    f"{PROJECT_DIR}/WUMI2024a_allsev",
    # add more reusable collections here
]

WALL_TO_WALL_CBI_COLLECTION = f"{PROJECT_DIR}/west_wall_to_wall_bccbi"
FOREST_MASKS_COLLECTION = f"{PROJECT_DIR}/forest_masks"

REQUESTED_BANDS = [
    #"CBI_bc",
    "dnbr",
    "rbr",
    "rdnbr",
    "dndvi",
    "devi",
    "dndmi",
    "dmirbi",
    "post_nbr",
    "post_mirbi",
    # "CBI_bc_masked_relaxed",
    # "CBI_bc_masked_conservative",
    # "CBI_bc_masked_rap10",
]

TARGET_CRS = "EPSG:5070"
TARGET_TRANSFORM = [30, 0, -2361600, 0, -30, 3177450]

RUN_YEARS = [2003, 2004, 2005]         # e.g. 1999, or [1999, 2000], or None to run all viable years
MAX_PIXELS = 1e13
SHARD_SIZE = 512
OVERWRITE_EXISTING = False

# -----------------------------------------------------------------------------
# INITIALIZE
# -----------------------------------------------------------------------------

ee.Authenticate(
    auth_mode="notebook",
    scopes=[
        "https://www.googleapis.com/auth/earthengine",
        "https://www.googleapis.com/auth/devstorage.read_write",
        "https://www.googleapis.com/auth/drive",
    ]
)
ee.Initialize(project=GEE_PROJECT)

# -----------------------------------------------------------------------------
# CONSTANTS
# -----------------------------------------------------------------------------

SPECIAL_MASKED_BANDS = [
    "CBI_bc_masked_relaxed",
    "CBI_bc_masked_conservative",
    "CBI_bc_masked_rap10",
]

COMPUTABLE_CORE_BANDS = [
    "CBI_bc",
    "dnbr",
    "rbr",
    "rdnbr",
    "dndvi",
    "devi",
    "dndmi",
    "dmirbi",
    "post_nbr",
    "post_mirbi",
]

ALL_SUPPORTED_BANDS = COMPUTABLE_CORE_BANDS + SPECIAL_MASKED_BANDS

WESTERN_STATES = [
    "WA", "OR", "CA", "ID", "NV", "MT", "WY", "UT", "CO", "AZ", "NM"
]

# -----------------------------------------------------------------------------
# SMALL CLIENT-SIDE HELPERS
# -----------------------------------------------------------------------------

def safe_description(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_-]", "_", name)[:100]


def asset_exists(asset_id: str) -> bool:
    try:
        ee.data.getAsset(asset_id)
        return True
    except Exception:
        return False


def ensure_image_collection(asset_id: str) -> None:
    if asset_exists(asset_id):
        meta = ee.data.getAsset(asset_id)
        if meta["type"] != "IMAGE_COLLECTION":
            raise ValueError(f"{asset_id} exists but is not an IMAGE_COLLECTION.")
        return

    ee.data.createAsset({"type": "IMAGE_COLLECTION"}, asset_id)
    print(f"Created image collection: {asset_id}")


def delete_asset_if_exists(asset_id: str) -> None:
    if asset_exists(asset_id):
        ee.data.deleteAsset(asset_id)


def validate_collection_has_year_property(collection_id: str) -> None:
    col = ee.ImageCollection(collection_id)
    n_images = col.size().getInfo()
    years = col.aggregate_array("year").getInfo()

    if len(years) != n_images:
        raise ValueError(
            f"{collection_id} failed validation: not all images contain a 'year' property."
        )


def validate_collection_year_uniqueness(collection_id: str) -> None:
    col = ee.ImageCollection(collection_id)
    years = col.aggregate_array("year").getInfo()
    if len(years) != len(set(years)):
        raise ValueError(
            f"{collection_id} failed validation: duplicate 'year' values found."
        )


def get_collection_band_names(collection_id: str) -> list[str]:
    col = ee.ImageCollection(collection_id)
    if col.size().getInfo() == 0:
        return []
    return ee.Image(col.first()).bandNames().getInfo()


def build_band_source_lookup(precomputed_collections: list[str]) -> dict[str, list[str]]:
    lookup: dict[str, list[str]] = {}
    for asset in precomputed_collections:
        band_names = get_collection_band_names(asset)
        for band in band_names:
            lookup.setdefault(band, []).append(asset)
    return lookup


# -----------------------------------------------------------------------------
# SERVER-SIDE HELPERS
# -----------------------------------------------------------------------------

def empty_masked_band(name: str) -> ee.Image:
    return ee.Image.constant(0).rename(name).toFloat().updateMask(ee.Image.constant(0))


def empty_masked_image(bands: list[str]) -> ee.Image:
    if not bands:
        raise ValueError("empty_masked_image requires at least one band name.")

    out = empty_masked_band(bands[0])
    for band in bands[1:]:
        out = out.addBands(empty_masked_band(band))
    return out


def select_and_rename(image: ee.Image, bands: list[str]) -> ee.Image:
    return ee.Image(image).select(bands, bands)


def get_year_band_from_collection(collection_id: str, year: int, band: str) -> ee.Image:
    col = ee.ImageCollection(collection_id).filter(ee.Filter.eq("year", year))

    return ee.Image(
        ee.Algorithms.If(
            col.size().gt(0),
            ee.Image(col.mosaic()).select([band]).rename([band]).toFloat(),
            empty_masked_band(band)
        )
    )


def build_reused_band(
    year: int,
    band: str,
    collection_ids: list[str],
    new_mask: ee.Image
) -> ee.Image:
    out = empty_masked_band(band)

    for collection_id in collection_ids:
        candidate = get_year_band_from_collection(collection_id, year, band)
        out = out.unmask(candidate)

    return out.updateMask(new_mask).rename(band).toFloat()


def build_wall_to_wall_cbi(
    year: int,
    new_mask: ee.Image
) -> ee.Image:
    cbi = get_year_band_from_collection(WALL_TO_WALL_CBI_COLLECTION, year, "CBI_bc")
    return cbi.updateMask(new_mask).rename("CBI_bc").toFloat()


def build_perimeter_mask(year_fires: ee.FeatureCollection, year_attr: str) -> ee.Image:
    return (
        year_fires.reduceToImage(
            properties=[year_attr],
            reducer=ee.Reducer.count()
        )
        .gt(0)
        .rename("perim_mask")
        .selfMask()
    )


def build_reused_core_image(
    year: int,
    core_bands: list[str],
    new_mask: ee.Image,
    band_source_lookup: dict[str, list[str]]
) -> ee.Image:
    out = None

    for band in core_bands:
        if band == "CBI_bc":
            band_img = build_wall_to_wall_cbi(year, new_mask)
        else:
            source_collections = band_source_lookup.get(band, [])
            band_img = build_reused_band(year, band, source_collections, new_mask)

        out = band_img if out is None else out.addBands(band_img)

    return out


def build_missing_any_mask(
    reused_core: ee.Image,
    core_bands: list[str],
    new_mask: ee.Image
) -> ee.Image:
    if len(core_bands) == 1:
        all_valid = reused_core.select(core_bands[0]).mask()
    else:
        all_valid = reused_core.select(core_bands).mask().reduce(ee.Reducer.min())

    return new_mask.And(all_valid.Not()).rename("missing_any").selfMask()


def fill_missing_from_computed(
    reused_core: ee.Image,
    computed_core: ee.Image,
    core_bands: list[str],
    new_mask: ee.Image
) -> ee.Image:
    out = None

    for band in core_bands:
        final_band = (
            reused_core.select(band)
            .unmask(computed_core.select(band))
            .updateMask(new_mask)
            .rename(band)
            .toFloat()
        )
        out = final_band if out is None else out.addBands(final_band)

    return out


def make_special_cbi_band(
    cbi: ee.Image,
    forest_mask: ee.Image,
    forest_band_name: str,
    out_name: str
) -> ee.Image:
    has_band = forest_mask.bandNames().contains(forest_band_name)

    return ee.Image(
        ee.Algorithms.If(
            has_band,
            cbi.updateMask(forest_mask.select(forest_band_name).gt(0)).rename(out_name).toFloat(),
            empty_masked_band(out_name)
        )
    )


def build_special_masked_bands(
    final_cbi: ee.Image,
    year: int
) -> ee.Image:
    prior_year = year - 1
    forest_img = ee.Image(
        ee.ImageCollection(FOREST_MASKS_COLLECTION)
        .filter(ee.Filter.eq("year", prior_year))
        .limit(1)
        .mosaic()
    )

    relaxed = make_special_cbi_band(
        cbi=final_cbi,
        forest_mask=forest_img,
        forest_band_name="relaxed",
        out_name="CBI_bc_masked_relaxed"
    )

    conservative = make_special_cbi_band(
        cbi=final_cbi,
        forest_mask=forest_img,
        forest_band_name="conservative",
        out_name="CBI_bc_masked_conservative"
    )

    rap10 = make_special_cbi_band(
        cbi=final_cbi,
        forest_mask=forest_img,
        forest_band_name="rap10",
        out_name="CBI_bc_masked_rap10"
    )

    return relaxed.addBands([conservative, rap10])


def count_unmasked_pixels(mask_img: ee.Image, region_fc: ee.FeatureCollection) -> ee.Number:
    stats = mask_img.reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=region_fc.geometry().bounds(),
        crs=TARGET_CRS,
        crsTransform=TARGET_TRANSFORM,
        maxPixels=MAX_PIXELS
    )
    return ee.Number(ee.Dictionary(stats).get(mask_img.bandNames().get(0), 0))


# -----------------------------------------------------------------------------
# PARKS-STYLE SEVERITY MODEL
# -----------------------------------------------------------------------------

rf_bands = ["def", "lat", "rbr", "dmirbi", "dndvi", "post_mirbi"]

cbi_training = ee.FeatureCollection(
    "users/grumpyunclesean/CBI_predictions/data_for_ee_model"
)

def_img = ee.Image("users/grumpyunclesean/CBI_predictions/def").rename("def").toInt()
lat = ee.Image.pixelLonLat().select("latitude").rename("lat").round().toInt()

nrow_training_fold = cbi_training.size()
minLeafPopulation = nrow_training_fold.divide(75).divide(6).round()

fsev_classifier = (
    ee.Classifier.smileRandomForest(
        numberOfTrees=500,
        minLeafPopulation=minLeafPopulation,
        seed=123
    )
    .train(cbi_training, "CBI", rf_bands)
    .setOutputMode("REGRESSION")
)

bandList = [
    "dnbr", "rbr", "rdnbr", "dndvi", "devi",
    "dndmi", "dmirbi", "post_nbr", "post_mirbi",
    "CBI", "CBI_bc"
]

ls9SR = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
ls8SR = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
ls7SR = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
ls5SR = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
ls4SR = ee.ImageCollection("LANDSAT/LT04/C02/T1_L2")


def applyScaleFactors(lsImage):
    opticalBands = lsImage.select(["SR_B."]).multiply(0.0000275).add(-0.2)
    return lsImage.addBands(opticalBands, None, True)


def ls8_9_Indices(lsImage):
    nbr = lsImage.normalizedDifference(["SR_B5", "SR_B7"]).toFloat()
    ndvi = lsImage.normalizedDifference(["SR_B5", "SR_B4"]).toFloat()
    ndmi = lsImage.normalizedDifference(["SR_B5", "SR_B6"]).toFloat()
    evi = lsImage.expression(
        "2.5 * ((SR_B5 - SR_B4) / (SR_B5 + 6 * SR_B4 - 7.5 * SR_B2 + 1))",
        {
            "SR_B5": lsImage.select("SR_B5"),
            "SR_B4": lsImage.select("SR_B4"),
            "SR_B2": lsImage.select("SR_B2"),
        }
    ).toFloat()
    mirbi = lsImage.expression(
        "((10 * SR_B6) - (9.8 * SR_B7) + 2)",
        {
            "SR_B6": lsImage.select("SR_B6"),
            "SR_B7": lsImage.select("SR_B7"),
        }
    ).toFloat()
    qa = lsImage.select(["QA_PIXEL"])

    return (
        nbr.addBands([ndvi, ndmi, evi, mirbi, qa])
        .select([0, 1, 2, 3, 4, 5], ["nbr", "ndvi", "ndmi", "evi", "mirbi", "QA_PIXEL"])
        .copyProperties(lsImage, ["system:time_start"])
    )


def ls4_7_Indices(lsImage):
    nbr = lsImage.normalizedDifference(["SR_B4", "SR_B7"]).toFloat()
    ndvi = lsImage.normalizedDifference(["SR_B4", "SR_B3"]).toFloat()
    ndmi = lsImage.normalizedDifference(["SR_B4", "SR_B5"]).toFloat()
    evi = lsImage.expression(
        "2.5 * ((SR_B4 - SR_B3) / (SR_B4 + 6 * SR_B3 - 7.5 * SR_B1 + 1))",
        {
            "SR_B4": lsImage.select("SR_B4"),
            "SR_B3": lsImage.select("SR_B3"),
            "SR_B1": lsImage.select("SR_B1"),
        }
    ).toFloat()
    mirbi = lsImage.expression(
        "((10 * SR_B5) - (9.8 * SR_B7) + 2)",
        {
            "SR_B5": lsImage.select("SR_B5"),
            "SR_B7": lsImage.select("SR_B7"),
        }
    ).toFloat()
    qa = lsImage.select(["QA_PIXEL"])

    return (
        nbr.addBands([ndvi, ndmi, evi, mirbi, qa])
        .select([0, 1, 2, 3, 4, 5], ["nbr", "ndvi", "ndmi", "evi", "mirbi", "QA_PIXEL"])
        .copyProperties(lsImage, ["system:time_start"])
    )


def lsCfmask(lsImage):
    cloudsBitMask = (1 << 3)
    cloudShadowBitMask = (1 << 4)
    snowBitMask = (1 << 5)
    waterBitMask = (1 << 7)

    qa = lsImage.select("QA_PIXEL")

    clear = (
        qa.bitwiseAnd(cloudsBitMask).eq(0)
        .And(qa.bitwiseAnd(cloudShadowBitMask).eq(0))
        .And(qa.bitwiseAnd(snowBitMask).eq(0))
        .And(qa.bitwiseAnd(waterBitMask).eq(0))
    )

    return (
        lsImage.updateMask(clear)
        .select([0, 1, 2, 3, 4])
        .copyProperties(lsImage, ["system:time_start"])
    )


waterMask = ee.Image("UMD/hansen/global_forest_change_2015").select(["datamask"]).eq(1)

ls9 = ls9SR.map(applyScaleFactors).map(ls8_9_Indices).map(lsCfmask)
ls8 = ls8SR.map(applyScaleFactors).map(ls8_9_Indices).map(lsCfmask)
ls7 = ls7SR.map(applyScaleFactors).map(ls4_7_Indices).map(lsCfmask)
ls5 = ls5SR.map(applyScaleFactors).map(ls4_7_Indices).map(lsCfmask)
ls4 = ls4SR.map(applyScaleFactors).map(ls4_7_Indices).map(lsCfmask)

lsCol = ee.ImageCollection(ls9.merge(ls8).merge(ls7).merge(ls5).merge(ls4))


def specImagery(regionF, fireYear, startday, endday, compute_mask):
    fireYear = ee.Date.parse("YYYY", fireYear)
    startday = ee.Number.parse(startday)
    endday = ee.Number.parse(endday)
    region = regionF.geometry().bounds()

    compute_mask = ee.Image(compute_mask).selfMask()

    preFireYear = fireYear.advance(-1, "year")
    preFireIndices = ee.Algorithms.If(
        lsCol.filterBounds(region)
        .filterDate(preFireYear, fireYear)
        .filter(ee.Filter.dayOfYear(startday, endday))
        .size(),
        lsCol.filterBounds(region)
        .filterDate(preFireYear, fireYear)
        .filter(ee.Filter.dayOfYear(startday, endday))
        .mean()
        .select([0, 1, 2, 3, 4], ["pre_nbr", "pre_ndvi", "pre_ndmi", "pre_evi", "pre_mirbi"]),
        ee.Image.cat(
            ee.Image(), ee.Image(), ee.Image(), ee.Image(), ee.Image()
        ).rename(["pre_nbr", "pre_ndvi", "pre_ndmi", "pre_evi", "pre_mirbi"])
    )

    preFireYear2 = fireYear.advance(-2, "year")
    preFireIndices2 = ee.Algorithms.If(
        lsCol.filterBounds(region)
        .filterDate(preFireYear2, fireYear)
        .filter(ee.Filter.dayOfYear(startday, endday))
        .size(),
        lsCol.filterBounds(region)
        .filterDate(preFireYear2, fireYear)
        .filter(ee.Filter.dayOfYear(startday, endday))
        .mean()
        .select([0, 1, 2, 3, 4], ["pre_nbr", "pre_ndvi", "pre_ndmi", "pre_evi", "pre_mirbi"]),
        ee.Image.cat(
            ee.Image(), ee.Image(), ee.Image(), ee.Image(), ee.Image()
        ).rename(["pre_nbr", "pre_ndvi", "pre_ndmi", "pre_evi", "pre_mirbi"])
    )

    pre_filled = (
        ee.Image(preFireIndices)
        .unmask(preFireIndices2)
        .updateMask(compute_mask)
    )

    postFireYear = fireYear.advance(1, "year")
    postFireIndices = ee.Algorithms.If(
        lsCol.filterBounds(region)
        .filterDate(postFireYear, fireYear.advance(2, "year"))
        .filter(ee.Filter.dayOfYear(startday, endday))
        .size(),
        lsCol.filterBounds(region)
        .filterDate(postFireYear, fireYear.advance(2, "year"))
        .filter(ee.Filter.dayOfYear(startday, endday))
        .mean()
        .select([0, 1, 2, 3, 4], ["post_nbr", "post_ndvi", "post_ndmi", "post_evi", "post_mirbi"]),
        ee.Image.cat(
            ee.Image(), ee.Image(), ee.Image(), ee.Image(), ee.Image()
        ).rename(["post_nbr", "post_ndvi", "post_ndmi", "post_evi", "post_mirbi"])
    )

    postFireIndices2 = ee.Algorithms.If(
        lsCol.filterBounds(region)
        .filterDate(postFireYear, fireYear.advance(3, "year"))
        .filter(ee.Filter.dayOfYear(startday, endday))
        .size(),
        lsCol.filterBounds(region)
        .filterDate(postFireYear, fireYear.advance(3, "year"))
        .filter(ee.Filter.dayOfYear(startday, endday))
        .mean()
        .select([0, 1, 2, 3, 4], ["post_nbr", "post_ndvi", "post_ndmi", "post_evi", "post_mirbi"]),
        ee.Image.cat(
            ee.Image(), ee.Image(), ee.Image(), ee.Image(), ee.Image()
        ).rename(["post_nbr", "post_ndvi", "post_ndmi", "post_evi", "post_mirbi"])
    )

    post_filled = (
        ee.Image(postFireIndices)
        .unmask(postFireIndices2)
        .updateMask(compute_mask)
    )

    fireIndices = pre_filled.addBands(post_filled)

    burnIndices = fireIndices.expression("(b('pre_nbr') - b('post_nbr')) * 1000") \
        .rename("dnbr").toInt().addBands(fireIndices)

    burnIndices2 = burnIndices.expression("b('dnbr') / (b('pre_nbr') + 1.001)") \
        .rename("rbr").toInt().addBands(burnIndices)

    burnIndices3 = burnIndices2.expression(
        "abs(b('pre_nbr')) < 0.001 ? 0.001 : b('pre_nbr')"
    ).abs().sqrt().rename("pre_nbr2").toFloat().addBands(burnIndices2)

    burnIndices4 = burnIndices3.expression("b('dnbr') / b('pre_nbr2')") \
        .rename("rdnbr").toInt().addBands(burnIndices3)

    burnIndices5 = burnIndices4.expression("(b('pre_ndvi') - b('post_ndvi')) * 1000") \
        .rename("dndvi").toInt().addBands(burnIndices4)

    burnIndices6 = burnIndices5.expression("(b('pre_evi') - b('post_evi')) * 1000") \
        .rename("devi").toInt().addBands(burnIndices5)

    burnIndices7 = burnIndices6.expression("(b('pre_ndmi') - b('post_ndmi')) * 1000") \
        .rename("dndmi").toInt().addBands(burnIndices6)

    burnIndices8 = burnIndices7.expression("(b('pre_mirbi') - b('post_mirbi')) * 1000") \
        .rename("dmirbi").toInt().addBands(burnIndices7)

    post_mirbi_1000 = burnIndices8.select("post_mirbi").multiply(1000).toInt()
    burnIndices8 = burnIndices8.addBands(post_mirbi_1000, None, True)

    burnIndices9 = burnIndices8.addBands(def_img.updateMask(compute_mask))
    burnIndices10 = burnIndices9.addBands(lat.updateMask(compute_mask)).updateMask(compute_mask)

    cbi_rf = (
        burnIndices10.select(rf_bands)
        .classify(fsev_classifier)
        .rename("CBI")
        .toFloat()
        .multiply(100).floor().divide(100)
        .updateMask(compute_mask)
    )

    burnIndices11 = cbi_rf.addBands(burnIndices10)

    def bias_correct(bandImage):
        cbi_lo = bandImage.expression("((b('CBI') - 1.5) * 1.3)  + 1.5")
        cbi_hi = bandImage.expression("((b('CBI') - 1.5) * 1.175) + 1.5")
        cbi_mg = bandImage.where(bandImage.lte(1.5), cbi_lo).where(bandImage.gt(1.5), cbi_hi)

        return (
            cbi_mg.where(cbi_mg.lt(0), 0)
            .where(cbi_mg.gt(3), 3)
            .multiply(100).floor().divide(100)
            .rename("CBI_bc")
        )

    validMask = pre_filled.select("pre_nbr").add(10).add(post_filled.select("post_nbr")).add(10)
    mask = waterMask.updateMask(validMask).updateMask(compute_mask)

    burnIndices12 = bias_correct(burnIndices11.select("CBI")).addBands(burnIndices11)
    burnIndices12 = (
        burnIndices12
        .select(bandList, bandList)
        .updateMask(mask)
        .updateMask(compute_mask)
        .clip(regionF)
    )

    return burnIndices12


def compute_year_core_image(
    year_fires: ee.FeatureCollection,
    year: int,
    compute_mask: ee.Image,
    core_bands: list[str]
) -> ee.Image:
    studyAreaNormalGrow = ee.FeatureCollection("TIGER/2018/States").filter(
        ee.Filter.inList(
            "NAME",
            [
                "Washington", "Oregon", "California", "Idaho", "Montana",
                "Nevada", "Colorado", "Wyoming", "Utah"
            ]
        )
    )

    studyAreaSpecialGrow = ee.FeatureCollection("TIGER/2018/States").filter(
        ee.Filter.inList("NAME", ["New Mexico", "Arizona"])
    )

    startDayNormal = ee.String("152")
    endDayNormal = ee.String("258")
    startDaySpecial = ee.String("91")
    endDaySpecial = ee.String("181")

    normal_fires = year_fires.filterBounds(studyAreaNormalGrow.geometry())
    special_fires = year_fires.filterBounds(studyAreaSpecialGrow.geometry())

    empty = empty_masked_image(core_bands)

    normal_compute_mask = compute_mask.updateMask(compute_mask).clip(normal_fires.geometry().bounds())
    special_compute_mask = compute_mask.updateMask(compute_mask).clip(special_fires.geometry().bounds())

    normal_img = ee.Image(
        ee.Algorithms.If(
            normal_fires.size().gt(0),
            specImagery(normal_fires, str(year), startDayNormal, endDayNormal, normal_compute_mask)
                .select(core_bands)
                .toFloat(),
            empty
        )
    )

    special_img = ee.Image(
        ee.Algorithms.If(
            special_fires.size().gt(0),
            specImagery(special_fires, str(year), startDaySpecial, endDaySpecial, special_compute_mask)
                .select(core_bands)
                .toFloat(),
            empty
        )
    )

    return ee.ImageCollection([normal_img, special_img]).mosaic().set({"year": year})


# -----------------------------------------------------------------------------
# PIPELINE LOGIC
# -----------------------------------------------------------------------------

unsupported = sorted(set(REQUESTED_BANDS) - set(ALL_SUPPORTED_BANDS))
if unsupported:
    raise ValueError(f"Unsupported requested bands: {unsupported}")

for collection_id in PRECOMPUTED_COLLECTIONS:
    validate_collection_has_year_property(collection_id)

validate_collection_has_year_property(WALL_TO_WALL_CBI_COLLECTION)
validate_collection_year_uniqueness(WALL_TO_WALL_CBI_COLLECTION)

validate_collection_has_year_property(FOREST_MASKS_COLLECTION)
validate_collection_year_uniqueness(FOREST_MASKS_COLLECTION)

band_source_lookup = build_band_source_lookup(PRECOMPUTED_COLLECTIONS)

fires_fc = ee.FeatureCollection(FIRES_ASSET)

fc_years = fires_fc.aggregate_array(YEAR_ATTR).distinct().sort().getInfo()
fc_years = sorted(int(y) for y in fc_years if y is not None)

if not fc_years:
    raise ValueError("No valid years found in the feature collection.")

min_fc_year = min(fc_years)
max_fc_year = max(fc_years)

start_year = max(min_fc_year, 1984)
now = datetime.now()
end_year = now.year - 1 if now.month >= 10 else now.year - 2
end_year = min(end_year, max_fc_year)

viable_years = list(range(start_year, end_year + 1))
invalid_fc_years = sorted(set(fc_years) - set(viable_years))
years_to_process = sorted(set(fc_years).intersection(viable_years))

if RUN_YEARS is not None:
    if isinstance(RUN_YEARS, int):
        years_to_process = [RUN_YEARS]
    else:
        years_to_process = RUN_YEARS

    # Validate all years
    invalid_years = [
        y for y in years_to_process
        if (y < start_year or y > end_year)
    ]

    if invalid_years:
        raise ValueError(
            f"RUN_YEARS contains invalid years {invalid_years}. "
            f"Valid range is {start_year}-{end_year}."
        )

print(f"Feature collection years: {fc_years}")
print(f"Viable Landsat years: {viable_years}")
print(f"Feature collection years outside viable Landsat range: {invalid_fc_years}")
print(f"Years to process: {years_to_process}")

output_collection_id = f"{PROJECT_DIR}/{PERIM_STRING}"
ensure_image_collection(output_collection_id)

states_fc = ee.FeatureCollection("TIGER/2018/States").filter(
    ee.Filter.inList("STUSPS", WESTERN_STATES)
)
export_region = states_fc.geometry()

requested_special_bands = [b for b in REQUESTED_BANDS if b in SPECIAL_MASKED_BANDS]
requested_non_special_bands = [b for b in REQUESTED_BANDS if b not in SPECIAL_MASKED_BANDS]

internal_core_bands = list(requested_non_special_bands)
if requested_special_bands and "CBI_bc" not in internal_core_bands:
    internal_core_bands = ["CBI_bc"] + internal_core_bands

internal_core_bands = [b for b in internal_core_bands if b in COMPUTABLE_CORE_BANDS]

for year in years_to_process:
    print(f"Preparing year {year}...")

    year_fires = fires_fc.filter(ee.Filter.eq(YEAR_ATTR, year))
    new_mask = build_perimeter_mask(year_fires, YEAR_ATTR)

    reused_core = build_reused_core_image(
        year=year,
        core_bands=internal_core_bands,
        new_mask=new_mask,
        band_source_lookup=band_source_lookup
    )

    missing_any = build_missing_any_mask(
        reused_core=reused_core,
        core_bands=internal_core_bands,
        new_mask=new_mask
    )

    missing_count = count_unmasked_pixels(missing_any, year_fires)

    # computed_core = ee.Image(
    #     ee.Algorithms.If(
    #         missing_count.gt(0),
    #         compute_year_core_image(
    #             year_fires=year_fires,
    #             year=year,
    #             compute_mask=missing_any,
    #             core_bands=internal_core_bands
    #         ),
    #         empty_masked_image(internal_core_bands)
    #     )
    # )

    computed_core = compute_year_core_image(
        year_fires=year_fires,
        year=year,
        compute_mask=missing_any,
        core_bands=internal_core_bands
    )

    final_core = fill_missing_from_computed(
        reused_core=reused_core,
        computed_core=computed_core,
        core_bands=internal_core_bands,
        new_mask=new_mask
    )

    final_image = final_core

    if requested_special_bands:
        final_cbi = final_core.select("CBI_bc")
        special_bands_img = build_special_masked_bands(final_cbi, year)
        final_image = final_image.addBands(special_bands_img)

    final_image = (
        final_image
        .select(REQUESTED_BANDS)
        .updateMask(new_mask)
        .toFloat()
        .set({
            "year": year,
            "perimeter_set": PERIM_STRING
        })
    )

    image_asset_id = f"{output_collection_id}/{PERIM_STRING}_{year}"

    if asset_exists(image_asset_id):
        if OVERWRITE_EXISTING:
            print(f"  deleting existing asset for {year}")
            delete_asset_if_exists(image_asset_id)
        else:
            print(f"  asset already exists for {year}; skipping")
            continue

    desc = safe_description(f"{PERIM_STRING}_{year}")

    task = ee.batch.Export.image.toAsset(
        image=final_image,
        description=desc,
        assetId=image_asset_id,
        region=export_region,
        crs=TARGET_CRS,
        crsTransform=TARGET_TRANSFORM,
        maxPixels=MAX_PIXELS,
        shardSize=SHARD_SIZE
    )
    task.start()
    print(f"  started export: {image_asset_id}")

print("All requested exports have been started.")

Feature collection years: [1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
Viable Landsat years: [1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
Feature collection years outside viable Landsat range: []
Years to process: [2003, 2004, 2005]
Preparing year 2003...
  started export: projects/ee-tymc5571-cbi-module/assets/welty_test_nocbi/welty_test_nocbi_2003
Preparing year 2004...
  started export: projects/ee-tymc5571-cbi-module/assets/welty_test_nocbi/welty_test_nocbi_2004
Preparing year 2005...
  started export: projects/ee-tymc5571-cbi-module/assets/welty_test_nocbi/welty_test_nocbi_2005
All requested exports have been started.
